# 02 · Feature Engineering

Build the sparse interaction matrix, user feature vectors, and item feature vectors used by all downstream models.

In [ ]:
import sys
sys.path.insert(0, '..')

import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz

from config import SPLITS_DIR, PROCESSED_DIR, REPORTS_DIR
from src.data.load_data import get_data
from src.data.preprocess import run_pipeline
from src.features.build_features import run_feature_pipeline

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1 · Run Preprocessing Pipeline

In [ ]:
train_raw, val_raw = get_data()
data = run_pipeline(train_raw, val_raw)

train    = data['train']
val      = data['val']
test     = data['test']
n_users  = data['n_users']
n_movies = data['n_movies']

print(f'\nTrain : {len(train):,} rows')
print(f'Val   : {len(val):,} rows')
print(f'Test  : {len(test):,} rows')
print(f'Users : {n_users:,}')
print(f'Movies: {n_movies:,}')

## 2 · Build Feature Matrices

In [ ]:
# Re-parse genre_list from string if needed (parquet round-trip)
for df in (train, val, test):
    if 'genre_list' in df.columns and isinstance(df['genre_list'].iloc[0], str):
        df['genre_list'] = df['genre_list'].apply(ast.literal_eval)

features = run_feature_pipeline(train, n_users, n_movies)

user_features   = features['user_features']
item_features   = features['item_features']
interaction_mat = features['interaction_mat']

## 3 · Interaction Matrix Analysis

In [ ]:
mat = interaction_mat
print(f'Matrix shape    : {mat.shape}')
print(f'Non-zero entries: {mat.nnz:,}')
print(f'Density         : {mat.nnz / (mat.shape[0] * mat.shape[1]):.4%}')
print(f'Sparsity        : {1 - mat.nnz / (mat.shape[0] * mat.shape[1]):.4%}')

# Ratings per user (non-zero rows)
user_nnz  = np.diff(mat.tocsr().indptr)
movie_nnz = np.diff(mat.tocsc().indptr)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(user_nnz[user_nnz > 0], bins=60, color='steelblue')
axes[0].set_title('Ratings per User (after filtering)')
axes[0].set_xlabel('# Ratings')

axes[1].hist(movie_nnz[movie_nnz > 0], bins=60, color='coral')
axes[1].set_title('Ratings per Movie (after filtering)')
axes[1].set_xlabel('# Ratings')

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'filtered_matrix_stats.png')
plt.show()

## 4 · User Feature Inspection

In [ ]:
print(f'User feature matrix: {user_features.shape}')
print('\nFeature columns:')
print(user_features.columns.tolist())
user_features.describe().round(3)

In [ ]:
# Distribution of mean ratings
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(user_features['mean_rating'], bins=50, color='steelblue')
axes[0].set_title('User Mean Rating Distribution')
axes[0].set_xlabel('Mean Rating')

axes[1].hist(user_features['rating_std'].clip(upper=2.5), bins=50, color='coral')
axes[1].set_title('User Rating Std Dev')
axes[1].set_xlabel('Std Dev')

axes[2].hist(user_features['log_n_ratings'], bins=50, color='mediumseagreen')
axes[2].set_title('User Activity (log-scale)')
axes[2].set_xlabel('log(1 + n_ratings)')

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'user_feature_distributions.png')
plt.show()

In [ ]:
# Genre affinity heatmap (sample users)
genre_cols = [c for c in user_features.columns if c.startswith('genre_affinity_')]
sample_uf  = user_features.sample(300, random_state=42).set_index('user_idx')[genre_cols]
sample_uf.columns = [c.replace('genre_affinity_', '') for c in sample_uf.columns]

plt.figure(figsize=(16, 5))
sns.heatmap(sample_uf.T, cmap='RdYlGn', center=3.5, linewidths=0,
            xticklabels=False, cbar_kws={'label': 'Avg Rating'})
plt.title('User Genre Affinity (sample of 300 users)')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'user_genre_affinity.png')
plt.show()

## 5 · Item Feature Inspection

In [ ]:
print(f'Item feature matrix: {item_features.shape}')
item_features.describe().round(3)

In [ ]:
# Year distribution
if 'year' in item_features.columns:
    years = item_features['year'].replace(0, np.nan).dropna()
    plt.figure(figsize=(10, 4))
    plt.hist(years, bins=50, color='mediumpurple')
    plt.title('Movie Release Year Distribution')
    plt.xlabel('Year')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'movie_years.png')
    plt.show()

In [ ]:
# Genre one-hot coverage
genre_ohe_cols = [c for c in item_features.columns if c.startswith('genre_') and c != 'genre_list']
genre_coverage = item_features[genre_ohe_cols].sum().sort_values(ascending=False)
genre_coverage.index = [c.replace('genre_', '') for c in genre_coverage.index]

plt.figure(figsize=(12, 4))
genre_coverage.plot(kind='bar', color='teal', edgecolor='white')
plt.title('Movies per Genre (one-hot encoding)')
plt.ylabel('# Movies')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'genre_onehot.png')
plt.show()

## 6 · Feature Correlation

In [ ]:
num_user_cols = ['mean_rating', 'rating_std', 'log_n_ratings']
corr = user_features[num_user_cols + genre_cols[:5]].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('User Feature Correlation (numeric + top-5 genres)')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'user_feature_corr.png')
plt.show()

## 7 · Summary

In [ ]:
print('=== Feature Engineering Summary ===')
print(f'Interaction matrix : {interaction_mat.shape}, sparsity {1 - interaction_mat.nnz/(interaction_mat.shape[0]*interaction_mat.shape[1]):.4%}')
print(f'User features      : {user_features.shape[0]} users × {user_features.shape[1]} features')
print(f'Item features      : {item_features.shape[0]} movies × {item_features.shape[1]} features')
print()
print('Key user features  : mean_rating, rating_std, log_n_ratings, genre_affinity_* (20)')
print('Key item features  : mean_rating, rating_std, log_n_ratings, year, genre_* (20)')
print()
print('All features saved to:', PROCESSED_DIR)